# Attention model training

Five attention-based variants are compared: three pure self-attention encoders, a CNN with learned attention pooling and a hierarchical patch transformer. Model selection and classification analysis use only train and tune; the test fold remains untouched.

## 1. Setup and fixed folds

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from functools import partial

from pytorch_timecourse_classification.analysis import (
    analyze_training_history, attention_rollout, evaluate_classifier,
    plot_attention_overlay, plot_history_comparison,
)
from pytorch_timecourse_classification.data import load_training_folds
from pytorch_timecourse_classification.experiments import (
    create_model, fit_model, save_and_verify_experiment,
)
from pytorch_timecourse_classification.models.attention import AttentionClassifier
from pytorch_timecourse_classification.models.cnn_attention_pooling import CNNAttentionPoolingClassifier
from pytorch_timecourse_classification.models.hierarchical_patch_transformer import HierarchicalPatchTransformerClassifier
from pytorch_timecourse_classification.training import TrainingConfig

In [ ]:
train_fold, tune_fold, preprocessor = load_training_folds()
class_labels = np.arange(len(preprocessor.class_names))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
for fold_name, fold in (("train", train_fold), ("tune", tune_fold)):
    counts = np.bincount(fold.targets.numpy(), minlength=len(class_labels))
    print(f"{fold_name:>5}: {tuple(fold.features.shape)}, {dict(zip(preprocessor.class_names, counts))}")
print(f"device: {device}")

In [ ]:
build_attention_model = partial(
    create_model,
    AttentionClassifier,
    input_length=train_fold.features.shape[-1],
    num_classes=len(preprocessor.class_names),
    random_seed=42,
)
build_model = partial(
    create_model,
    input_length=train_fold.features.shape[-1],
    num_classes=len(preprocessor.class_names),
    random_seed=42,
)
fit = partial(
    fit_model,
    training_fold=train_fold,
    tuning_fold=tune_fold,
    device=device,
)
evaluate_tune = partial(
    evaluate_classifier,
    fold=tune_fold,
    class_names=preprocessor.class_names,
    device=device,
)
save_and_verify = partial(
    save_and_verify_experiment,
    preprocessor=preprocessor,
    training_fold=train_fold,
    tuning_fold=tune_fold,
    device=device,
)

## 2. Variant A — compact mean-pooling attention

### 2.1 Model generation

A shallow pre-norm encoder is the low-variance reference. Weighted loss gives the two less frequent edge doses more influence.

In [ ]:
compact_model, compact_parameter_count = build_attention_model(
    embedding_dim=32, num_heads=4, num_layers=2, feedforward_dim=96,
    dropout=0.15, pooling="mean", norm_first=True,
)
compact_training_config = TrainingConfig(
    epochs=120, batch_size=64, learning_rate=5e-4, weight_decay=1e-4,
    scheduler_step_size=15, learning_rate_decay=0.80, patience=18,
    class_balance_strategy="weighted_loss",
)

### 2.2 Training

In [ ]:
compact_history = fit(compact_model, compact_training_config)

### 2.3 History analysis

In [ ]:
compact_history_summary = analyze_training_history(compact_history)

### 2.4 Tune confusion analysis

In [ ]:
compact_tune_metrics = evaluate_tune(compact_model)

### 2.5 Saving artifacts

In [ ]:
compact_reloaded, compact_paths = save_and_verify("attention_compact", compact_model, compact_history)

## 3. Variant B — regularized baseline attention

### 3.1 Model generation

The baseline retains a 64-dimensional representation but uses a wider feed-forward block, pre-norm layers and stronger optimizer regularization.

In [ ]:
baseline_model, baseline_parameter_count = build_attention_model(
    embedding_dim=64, num_heads=4, num_layers=3, feedforward_dim=192,
    dropout=0.20, pooling="mean", norm_first=True,
)
baseline_training_config = TrainingConfig(
    epochs=140, batch_size=48, learning_rate=3e-4, weight_decay=2e-4,
    scheduler_step_size=15, learning_rate_decay=0.80, patience=20,
    class_balance_strategy="weighted_loss",
)

### 3.2 Training

In [ ]:
baseline_history = fit(baseline_model, baseline_training_config)

### 3.3 History analysis

In [ ]:
baseline_history_summary = analyze_training_history(baseline_history)

### 3.4 Tune confusion analysis

In [ ]:
baseline_tune_metrics = evaluate_tune(baseline_model)

### 3.5 Saving artifacts

In [ ]:
baseline_reloaded, baseline_paths = save_and_verify("attention_baseline", baseline_model, baseline_history)

## 4. Variant C — wider CLS-token attention

### 4.1 Model generation

A learned CLS token lets attention construct a dedicated global summary instead of averaging every time point. The balanced sampler tests an alternative to loss weighting.

In [ ]:
cls_model, cls_parameter_count = build_attention_model(
    embedding_dim=96, num_heads=8, num_layers=4, feedforward_dim=384,
    dropout=0.25, pooling="cls", norm_first=True,
)
cls_training_config = TrainingConfig(
    epochs=160, batch_size=32, learning_rate=2e-4, weight_decay=5e-4,
    scheduler_step_size=20, learning_rate_decay=0.80, patience=22,
    class_balance_strategy="balanced_sampler",
)

### 4.2 Training

In [ ]:
cls_history = fit(cls_model, cls_training_config)

### 4.3 History analysis

In [ ]:
cls_history_summary = analyze_training_history(cls_history)

### 4.4 Tune confusion analysis

In [ ]:
cls_tune_metrics = evaluate_tune(cls_model)

### 4.5 Saving artifacts

In [ ]:
cls_reloaded, cls_paths = save_and_verify("attention_cls", cls_model, cls_history)

## 5. Variant D — CNN with attention pooling

A convolutional encoder first converts local trajectory motifs into a shorter token sequence. A learned scalar attention score then weights these tokens before classification. This combines the locality and shift tolerance of CNN features with an interpretable, data-dependent temporal pooling step.

```text
time course → strided CNN → temporal feature tokens → softmax attention weights
                                                   ↓ weighted sum → classifier
```

### 5.1 Model generation

In [ ]:
cnn_pooling_model, cnn_pooling_parameter_count = build_model(
    CNNAttentionPoolingClassifier, channels=(32, 64, 96),
    kernel_sizes=(9, 7, 5), strides=(2, 2, 1),
    attention_hidden_dim=64, classifier_hidden_dim=64, dropout=0.30,
)
cnn_pooling_training_config = TrainingConfig(
    epochs=140, batch_size=128, learning_rate=4e-4, weight_decay=5e-4,
    scheduler_step_size=15, learning_rate_decay=0.80, patience=20,
    class_balance_strategy="weighted_loss",
)

### 5.2 Training

In [ ]:
cnn_pooling_history = fit(cnn_pooling_model, cnn_pooling_training_config)

### 5.3 History analysis

If this hybrid converges faster or generalizes better than point-wise self-attention, the convolutional locality prior is useful. A very peaked pooling distribution with a widening train–tune gap would suggest that the model is memorizing a few unstable locations.

In [ ]:
cnn_pooling_history_summary = analyze_training_history(cnn_pooling_history)

### 5.4 Tune confusion analysis

In [ ]:
cnn_pooling_tune_metrics = evaluate_tune(cnn_pooling_model)

### 5.5 Saving artifacts

In [ ]:
cnn_pooling_reloaded, cnn_pooling_paths = save_and_verify(
    "attention_cnn_pooling", cnn_pooling_model, cnn_pooling_history
)

## 6. Variant E — hierarchical patch transformer

Non-overlapping temporal patches replace individual time points as tokens. A first transformer models relations between local patches; neighboring representations are then merged and a second transformer models the trajectory at a coarser scale. This reduces the attention sequence length and explicitly separates local from long-range structure.

```text
time course → patch embedding → fine transformer → merge neighboring patches
                                                   ↓
                              coarse transformer → mean pool → classifier
```

### 6.1 Model generation

In [ ]:
hierarchical_model, hierarchical_parameter_count = build_model(
    HierarchicalPatchTransformerClassifier, patch_size=8,
    embedding_dim=48, num_heads=4, stage1_layers=2, stage2_layers=2,
    feedforward_multiplier=3, dropout=0.20, norm_first=True,
)
hierarchical_training_config = TrainingConfig(
    epochs=150, batch_size=64, learning_rate=3e-4, weight_decay=5e-4,
    scheduler_step_size=18, learning_rate_decay=0.80, patience=22,
    class_balance_strategy="weighted_loss",
)

### 6.2 Training

In [ ]:
hierarchical_history = fit(hierarchical_model, hierarchical_training_config)

### 6.3 History analysis

A gain over full-resolution attention would indicate that patch-level structure is sufficient and the hierarchy regularizes useful long-range interactions. Underfitting may mean that patches are too wide; overfitting may require fewer layers or a smaller second-stage embedding.

In [ ]:
hierarchical_history_summary = analyze_training_history(hierarchical_history)

### 6.4 Tune confusion analysis

In [ ]:
hierarchical_tune_metrics = evaluate_tune(hierarchical_model)

### 6.5 Saving artifacts

In [ ]:
hierarchical_reloaded, hierarchical_paths = save_and_verify(
    "attention_hierarchical_patch", hierarchical_model, hierarchical_history
)

## 7. Direct comparison

Use tune macro-F1 as the primary metric and tune loss as a stability check. The larger model is justified only if its improvement survives repeated seeds.

In [ ]:
comparison = pd.DataFrame([
    {"variant": "compact", "parameters": compact_parameter_count, **compact_history_summary, **compact_tune_metrics},
    {"variant": "baseline", "parameters": baseline_parameter_count, **baseline_history_summary, **baseline_tune_metrics},
    {"variant": "cls", "parameters": cls_parameter_count, **cls_history_summary, **cls_tune_metrics},
    {"variant": "cnn_pooling", "parameters": cnn_pooling_parameter_count, **cnn_pooling_history_summary, **cnn_pooling_tune_metrics},
    {"variant": "hierarchical", "parameters": hierarchical_parameter_count, **hierarchical_history_summary, **hierarchical_tune_metrics},
]).set_index("variant")
comparison.round(4)

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
comparison[["tune_accuracy", "tune_macro_f1"]].plot.bar(ax=axes[0], ylim=(0, 1), rot=30)
axes[0].set(title="Tune classification metrics", ylabel="score")
comparison["parameters"].plot.bar(ax=axes[1], color="0.4", rot=30)
axes[1].set(title="Trainable parameters", ylabel="parameters")
for axis in axes:
    axis.grid(axis="y", alpha=0.2)
figure.tight_layout()

## 8. Project attention back to the time courses

Raw attention weights are not automatically faithful feature attributions. As a diagnostic, attention rollout averages heads, includes residual connections and composes attention across layers. For CLS pooling, the CLS-to-timepoint influence is used; for mean pooling, influence is averaged over output timepoints. The resulting values are overlaid on representative tune trajectories and summarized per class. This rollout comparison is restricted to the three full-resolution self-attention variants because CNN pooling weights and hierarchical stage attentions have different meanings and resolutions.

In [ ]:
attention_models = {"compact": compact_model, "baseline": baseline_model, "cls": cls_model}
rollout_variants = list(attention_models)
best_variant = comparison.loc[rollout_variants, "tune_macro_f1"].idxmax()
analysis_model = attention_models[best_variant]
analysis_model.eval()
with torch.no_grad():
    tune_predictions = analysis_model(tune_fold.features.to(device)).argmax(dim=1).cpu()
print(f"attention analysis uses tune-selected variant: {best_variant}")

In [ ]:
example_indices = []
for class_index in class_labels:
    class_indices = torch.where(tune_fold.targets == class_index)[0]
    correct_indices = class_indices[tune_predictions[class_indices] == class_index]
    example_indices.append(int(correct_indices[0] if len(correct_indices) else class_indices[0]))

with torch.no_grad():
    _, example_attention = analysis_model.forward_with_attention(
        tune_fold.features[example_indices].to(device)
    )
    example_importance = attention_rollout(
        example_attention, pooling=analysis_model.pooling
    ).cpu().numpy()

figure, axes = plt.subplots(2, 3, figsize=(15, 7), sharex=True)
for axis, sample_index, importance in zip(axes.flat, example_indices, example_importance):
    true_name = preprocessor.class_names[int(tune_fold.targets[sample_index])]
    predicted_name = preprocessor.class_names[int(tune_predictions[sample_index])]
    plot_attention_overlay(
        tune_fold.trajectories[sample_index], importance, axis=axis,
        title=f"true: {true_name}; predicted: {predicted_name}",
    )
figure.suptitle(f"Tune trajectories with attention rollout — {best_variant}")
figure.tight_layout()

In [ ]:
rollout_batches = []
with torch.no_grad():
    for start in range(0, len(tune_fold.features), 16):
        _, batch_attention = analysis_model.forward_with_attention(
            tune_fold.features[start:start + 16].to(device)
        )
        rollout_batches.append(
            attention_rollout(batch_attention, pooling=analysis_model.pooling).cpu()
        )
all_importance = torch.cat(rollout_batches).numpy()
class_mean_importance = np.vstack([
    all_importance[tune_fold.targets.numpy() == class_index].mean(axis=0)
    for class_index in class_labels
])
figure, axis = plt.subplots(figsize=(12, 3.5))
image = axis.imshow(class_mean_importance, aspect="auto", cmap="magma")
axis.set(yticks=class_labels, yticklabels=preprocessor.class_names, xlabel="time index", title="Mean attention rollout per true tune class")
figure.colorbar(image, ax=axis, label="mean rollout importance")
figure.tight_layout()

## 9. Interpretation and next steps

Interpret the variants in two stages. First inspect train and tune loss: high, parallel plateaus indicate underfitting or unsuccessful optimization, whereas a falling train loss combined with a rising tune loss and a widening accuracy gap indicates overfitting. The compact model is useful as a variance-controlled reference. The baseline should improve tune macro-F1 without opening a large gap. The CLS model is worthwhile only if its learned global summary improves tune performance rather than training performance alone. CNN attention pooling tests whether a local convolutional prior is more sample-efficient, while the hierarchical transformer tests whether patching and coarse-to-fine attention improve long-range modeling.

Then inspect the confusion matrices. Weighted loss and balanced sampling are successful only if they recover the less frequent 0 pM and 100 pM classes without broadly degrading the intermediate doses. Macro-F1 is therefore more informative than accuracy for model selection. Tune loss remains important because two models with similar accuracy can differ substantially in confidence and calibration.

Next, repeat each variant with several seeds and report mean and standard deviation of tune macro-F1. For CNN pooling, project its token weights back to time intervals and check stability across seeds. For the hierarchical transformer, vary patch size and compare fine-stage with coarse-stage attention rather than mixing their resolutions. If only the CLS model overfits, reduce embedding/feed-forward width or increase weight decay and dropout. Compare attention-based views against gradient attribution or input occlusion: stable agreement is stronger evidence than one attractive attention plot. Further useful checks are probability calibration and group-aware folds if trajectories from one experiment or replicate are correlated. Select the family winner on tune only; use test once in the subsequent comparison notebook.

## 10. Training-history comparison

The four panels compare train and tune loss and accuracy for all five attention variants. Each dot marks the checkpoint epoch restored after early stopping.

In [ ]:
history_comparison_figure = plot_history_comparison({
    "compact": compact_history,
    "baseline": baseline_history,
    "cls": cls_history,
    "cnn_pooling": cnn_pooling_history,
    "hierarchical": hierarchical_history,
})